# Quantum Teleportation

As the name suggests, it is like assuming a qubit teleports itself from one user to another over a classical channel. The word teleportation is used because what essentially travels is the classical bits or values because they communicate over a classical channel. So, for a layman, it almost feels like the information of the qubit has travelled to Bob without having moved from Alice. This somehow makes sense because both Alice and Bob share a Bell state between them (which is an entangled state). 

The key-insight is however the fact that as mentioned above, it is not essentially a teleportation because nothing can travel faster than the speed of light. Alice's qubit is destroyed the moment she performs a Bell measurement on her qubit. Based only on the classical bits transferred from Alice to Bob, he is able to retrieve her original state by doing an appropriate X or Z gate operation on his half based on the two classical bits he received from her.

The key steps involved in this are as follows-

1. Alice prepares a qubit from her end. She already has her share of the entangled state (Bell state) which is common with Bob.
2. Alice then performs a Bell Measurement on her end to get the classical bits for her to share it to Bob.
3. The classical bits are then sent to Bob via a classical channel.
4. Using the classical bits, Bob uses an appropriate gate operation on his end of the entangled state to get Alice's qubit.

In [3]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

In [9]:
circuit=QuantumCircuit(3,3)

circuit.h(0)  #superposition on qubit 1

circuit.h(1)
circuit.cx(1,2)  #entanglement between qubits 2 and 3

#Alice's Bell measurement

circuit.cx(0,1)
circuit.h(0)

circuit.measure(0,0)  #measuring qubit 1
circuit.measure(1,1)  #measuring qubit 2

In [11]:
#Bob's measurement

circuit.measure(2,2) #measuring qubit 3

#Circuit

simulator = AerSimulator()

job = simulator.run(circuit, shots=1000)

result = job.result()

counts = result.get_counts()

print(counts)

{'011': 118, '111': 143, '110': 113, '100': 134, '001': 122, '000': 136, '101': 113, '010': 121}


In [13]:
#Analysis- we check if the corrections reveal the correct teleported state

for outcome, count in sorted(counts.items()):
    q0_bit = int(outcome[2])
    q1_bit = int(outcome[1])
    q2_bit = int(outcome[0])
    
    corrected_q2 = q2_bit
    if q1_bit == 1:
        corrected_q2 ^= 1
    
    print(f"Alice: q0={q0_bit} q1={q1_bit} | Bob raw: {q2_bit} | Bob corrected: {corrected_q2} | Count: {count}")

Alice: q0=0 q1=0 | Bob raw: 0 | Bob corrected: 0 | Count: 136
Alice: q0=1 q1=0 | Bob raw: 0 | Bob corrected: 0 | Count: 122
Alice: q0=0 q1=1 | Bob raw: 0 | Bob corrected: 1 | Count: 121
Alice: q0=1 q1=1 | Bob raw: 0 | Bob corrected: 1 | Count: 118
Alice: q0=0 q1=0 | Bob raw: 1 | Bob corrected: 1 | Count: 134
Alice: q0=1 q1=0 | Bob raw: 1 | Bob corrected: 1 | Count: 113
Alice: q0=0 q1=1 | Bob raw: 1 | Bob corrected: 0 | Count: 113
Alice: q0=1 q1=1 | Bob raw: 1 | Bob corrected: 0 | Count: 143


The raw data for Bob is the measurement of Bob's qubit before Alice sent her classical bits which may include raw noise and hence its sometimes 0 and sometimes 1. Through this we see that there's a 50/50 chance that Bob had to flip his qubit based on what Alice measured on her half of the entangled state. Whenever Alice measured q1 as 1, provided the raw data sent to Bob was 0, this meant her qubit got flipped at Bob's end for which Bob had to correct and flip it back.

But, this still doesn't show with 100% accuracy if Quantum teleportation was applied successfully or not because of the 50/50 superposition. Let us now show with certainty that the Quantum Teleportation protocol is legit for a definitive state.

### Restart the kernel before compiling the definitive state case.

In [12]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

circuit=QuantumCircuit(3,3)

circuit.x(0) #first qubit is now a definitive state of |1>

circuit.h(1) #second qubit
circuit.cx(1,2) #entanglement between qubits 2 and 3

#Alice's measurement

circuit.cx(0,1)
circuit.h(0)

circuit.measure(0,0)
circuit.measure(1,1)
circuit.measure(2,2) #Bob's measurement

#Circuit

simulator = AerSimulator()

job = simulator.run(circuit, shots=1000)

result = job.result()

counts = result.get_counts()

print(counts)

#Analysis- we check if the corrections reveal the correct teleported state

for outcome, count in sorted(counts.items()):
    q0_bit = int(outcome[2])
    q1_bit = int(outcome[1])
    q2_bit = int(outcome[0])
    
    corrected_q2 = q2_bit
    if q1_bit == 1:
        corrected_q2 ^= 1
    
    print(f"Alice: q0={q0_bit} q1={q1_bit} | Bob raw: {q2_bit} | Bob corrected: {corrected_q2} | Count: {count}")

print(circuit.draw())

{'010': 270, '101': 230, '100': 239, '011': 261}
Alice: q0=0 q1=1 | Bob raw: 0 | Bob corrected: 1 | Count: 270
Alice: q0=1 q1=1 | Bob raw: 0 | Bob corrected: 1 | Count: 261
Alice: q0=0 q1=0 | Bob raw: 1 | Bob corrected: 1 | Count: 239
Alice: q0=1 q1=0 | Bob raw: 1 | Bob corrected: 1 | Count: 230
     ┌───┐          ┌───┐┌─┐
q_0: ┤ X ├───────■──┤ H ├┤M├
     ├───┤     ┌─┴─┐└┬─┬┘└╥┘
q_1: ┤ H ├──■──┤ X ├─┤M├──╫─
     └───┘┌─┴─┐└┬─┬┘ └╥┘  ║ 
q_2: ─────┤ X ├─┤M├───╫───╫─
          └───┘ └╥┘   ║   ║ 
c: 3/════════════╩════╩═══╩═
                 2    1   0 


For a definitive state as above, we can see that the corrected qubit for Bob is always 1 which matches with the qubit that Alice initially had.

When Alice measured q1 as 1 and Bob had raw data as 0, he flipped it to get 1. When Alice measured q1 as 0 and Bob already had 1 as his qubit, he didn't have to flip it and hence his corrected state was already the same as that of Alice's original qubit.

Here, what needs to be accounted for is that when Alice measured her qubit with her half of the entangled state, her original qubit got destroyed and she was left with the measured classical bits. She then sent those classical bits over to Bob based on which he applied the appropriate gate operations and then measured. So, the information was shared at the speed of light and not faster than that.